# Linear ideal stability analysis with DCON

Is a given VEST equilibrium stable to ideal external kinks, and if not, which
toroidal mode goes first?

DCON answers that by computing the perturbed potential energy delta-W for each
toroidal mode number n. A positive total energy means the equilibrium is ideally
stable to that mode; a negative one means it is not, and the eigenfunction says
where the displacement lives.

This notebook runs DCON on an equilibrium stored in the repository, maps the
result into the `mhd_linear` IDS, and draws it through the plot catalog. Three
kinds of number are kept apart, as elsewhere in VAFT:

| | |
| --- | --- |
| **Machine description** | the VEST wall and coil geometry DCON's vacuum calculation uses |
| **Modelling input** | the equilibrium, and the `dcon.in` truncation and grid settings |
| **Solver output** | delta-W, the marginal points, the eigenfunctions |

The equilibrium is a reconstruction, not a measurement of stability, so what
follows is the stability of a *reconstructed* VEST equilibrium.

## 0. Setup

DCON is part of the GPEC suite, which VAFT does not ship and cannot install --
it is obtained from its maintainers under their own terms. VAFT resolves
`$GPECHOME/bin/dcon`, the same `$XHOME` convention CHEASE, EFIT, NUBEAM and TES
follow. Without it every section below reports what it would show and stops.

Everything else is in the repository. The namelist templates `dcon.in`,
`equil.in` and `vac.in` are packaged in `vaft/data/gpec` and ship in the wheel;
the equilibrium is `vaft/data/efit/g039915.00319`, which is repository-only.
The run goes into a temporary directory and section 7 removes it.

`notebooks/initialize_external_fusion_codes.ipynb` reports every configured
root at once if you would rather check the whole environment first.

In [ ]:
import contextlib
import os
import tempfile
import time
from pathlib import Path

try:
    _ipython = get_ipython()
except NameError:
    _ipython = None
if _ipython is None:
    os.environ.setdefault("MPLBACKEND", "Agg")
elif "IPKernelApp" in _ipython.config:
    _ipython.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
import numpy as np

import vaft
from vaft.code import gpec
from vaft.code._executables import executable_from_home, missing_home_message
from vaft.data.resources import data_path

plt.rcParams["figure.dpi"] = 110

SHOT = 39915
TIME_MS = 319
GEQDSK = data_path(f"efit/g0{SHOT}.00{TIME_MS}")

# The toroidal modes to solve. This is the adapter's own default: n=1 is the
# mode that limits a spherical tokamak in practice, and n=2 is the cheapest
# check that the answer is not an artefact of one mode. Higher n costs
# disproportionately more -- n=3 took 97 s against 8 s for n=1 on this
# equilibrium -- so add them deliberately rather than by habit.
MODES = (1, 2)

CONFIG = gpec.GPECSuiteConfig(modules=("dcon",), modes=MODES)
EXECUTABLE = executable_from_home(
    os.environ.get(gpec.GPEC_HOME_ENV),
    home_variable=gpec.GPEC_HOME_ENV,
    relative_path="bin/dcon",
    code_name="DCON",
)
HAVE_DCON = EXECUTABLE is not None

print(f"equilibrium {GEQDSK.name}  ({'found' if GEQDSK.is_file() else 'MISSING'})")
print(f"templates   {data_path('gpec')}")
print(f"modes       {', '.join(f'n={n}' for n in MODES)}")
print(f"executable  {EXECUTABLE}")
if not HAVE_DCON:
    # The adapters raise this exact text, so the notebook and a failing run
    # say the same thing rather than two different things.
    print()
    print(missing_home_message(
        home_variable=gpec.GPEC_HOME_ENV,
        relative_path="bin/dcon",
        code_name="DCON",
    ))
    print("\nThe GPEC-suite build itself is tracked in issue #226.")

## 1. Prepare and run

`run_gpec_suite_case` writes one directory per (time, module, mode) --
`{workdir}/{time}/{module}/nn={n}/` -- fills each with the equilibrium and the
three namelists, and runs the solver there. The directory grammar matters
later: section 4 reads the results back through the same helper the adapter
uses to write them, so the two cannot drift apart.

Nothing here is a reduced configuration. The packaged `dcon.in` is used as it
is, which is what makes the delta-W below a result rather than a demonstration.

In [ ]:
_cleanup = contextlib.ExitStack()
WORKDIR = None
result = None

if HAVE_DCON:
    WORKDIR = Path(
        _cleanup.enter_context(tempfile.TemporaryDirectory(prefix="vaft-dcon-"))
    )
    inputs = gpec.GPECCaseInputs(
        shot=SHOT, time_ms=TIME_MS, geqdsk=GEQDSK, workdir=WORKDIR
    )
    print(f"work directory  {WORKDIR}")

    _started = time.monotonic()
    result = gpec.run_gpec_suite_case(inputs, CONFIG)
    print(f"elapsed         {time.monotonic() - _started:.0f} s")
    print()
    for record in result.records:
        print(f"  dcon n={record.mode}  {record.status:9s} rc={record.returncode}"
              f"  {len(record.outputs)} output files")
        if record.reason:
            print(f"      {record.reason}")

HAVE_RUN = result is not None and result.returncode == 0
if not HAVE_DCON:
    print(
        "Would prepare one directory per toroidal mode, run DCON in each, and\n"
        "report the status and outputs of every cell."
    )

## 2. What DCON produced

Each mode directory holds DCON's native output: `dcon.out` (the human-readable
log, including the energy table), `dcon_output_n<N>.nc` (the netCDF VAFT
reads), `solutions.bin` (the eigenfunctions), and the Euler-Lagrange
integration intermediates.

Listing them is not busywork. A cell that ran but produced no `.nc` is the
failure mode that otherwise surfaces much later as an empty IDS, and
`validate_dcon_result` exists precisely because "the executable returned 0" and
"there is a usable result" are different claims.

In [ ]:
if HAVE_RUN:
    for record in result.records:
        problems = gpec.validate_dcon_result(record.workdir, record.mode)
        verdict = "usable" if not problems else "UNUSABLE -- " + "; ".join(problems)
        print(f"n={record.mode}: {verdict}")
        names = sorted(p.name for p in record.workdir.iterdir())
        print(f"   {len(names)} files: {', '.join(names[:9])}"
              f"{', ...' if len(names) > 9 else ''}")
        if record.missing_optional_outputs:
            print(f"   no companion output: {record.missing_optional_outputs}")
else:
    print("Would check each mode directory for a usable DCON result.")

## 3. The ideal stability verdict

DCON returns a whole spectrum of energy eigenvalues per mode -- one per
eigenmode of the perturbed-energy operator, sorted ascending -- and the first
is the least stable. Its sign is the verdict:

- **positive** -- no ideal mode with this n can lower the plasma's energy, so
  the equilibrium is ideally stable to it;
- **negative** -- an ideal instability exists, and its growth is not limited by
  resistivity.

The magnitude is *not* a growth rate and not an energy in joules. DCON
normalizes its eigenvector arbitrarily, so the eigenvalue is dimensionless and
only its sign, and the ordering between modes, carry meaning. This is the same
caveat the mapping records in `code.parameters` when it writes the value into
`energy_perturbed`, a field IMAS documents as joules.

DCON also reports separate plasma-only and vacuum eigenvalue spectra. They are
deliberately not shown as a decomposition here: the three spectra are
diagonalized separately, so the plasma and vacuum entries at a given index do
not add up to the total at that index, and presenting them side by side would
invite exactly that reading.

In [ ]:
if HAVE_RUN:
    for record in result.records:
        native = gpec.read_dcon_output(record.workdir, mode=record.mode)
        # DCON returns the whole spectrum, ascending, so element 0 is the
        # least-stable mode -- the one that decides the verdict, and the one
        # the IMAS mapping stores.
        spectrum = np.asarray(native.W_t_eigenvalue).real
        least = float(spectrum[0])
        unstable = int((spectrum < 0).sum())
        verdict = "ideally STABLE" if least > 0 else "ideally UNSTABLE"
        print(f"n={record.mode}:  least-stable delta-W {least:+.4g}   {verdict}")
        print(f"      {spectrum.size} eigenvalues, {unstable} negative"
              f";  dominant harmonic m={native.m_pol_dominant}")
    print(f"\nlimiting surface: q={float(native.qlim):.3f}"
          f" at psi_n={float(native.psilim):.4f}")
else:
    print("Would report the delta-W eigenvalue and its sign for each mode.")

## 4. Into IMAS

`build_mhd_linear_ods` reads the run directory back and populates `mhd_linear`.
It walks the same `{time}/{module}/nn={n}/` grammar the adapter wrote, computed
by the same helper rather than re-derived here.

What reaches the IDS is deliberately less than what DCON computed. `n_tor`, the
energy eigenvalue with its normalization caveat, the dominant poloidal harmonic
and a run-success `code.output_flag` have exact homes. The Fourier-space
eigenfunction reaches `plasma.displacement_perpendicular` on an explicitly
declared `(psi, m)` grid as a *closest-fit* mapping, with the discrepancy
recorded in `code.parameters` rather than smoothed over. Quantities with no
correct home are left in the native containers instead of being forced into an
approximate field.

In [ ]:
ods = None
manifest = None

if HAVE_RUN:
    from vaft.omas.vest_upstream import build_mhd_linear_ods

    ods, manifest = build_mhd_linear_ods(
        shot=SHOT, time_values=[TIME_MS], workdir=WORKDIR,
        modules=("dcon",), modes=MODES,
    )
    print(f"manifest status: {manifest['status']}")
    for cell, record in manifest["modules_modes"].items():
        for mode, values in record.get("modes", {}).items():
            print(f"  {cell}: {values['variable']} = {values['value']:+.4g}")

    print()
    written = [p for p in sorted(ods.flat()) if p.startswith("mhd_linear.time_slice")]
    print(f"{len(written)} mhd_linear.time_slice paths written, e.g.")
    for path in written[:6]:
        print(f"  {path}")
else:
    print("Would map the run into mhd_linear and list what was written.")

## 5. The registered plots

Mapped, the result is drawable through the plot catalog rather than by reaching
into DCON's files. Each of these is a registered recipe, so the same call works
on an ODS reloaded from disk long after the run directory is gone.

`plot_mhd_linear_profile_displacement` defaults to the **least stable** cell it
can find rather than the first one, which is the cell you would have picked by
hand. Harmonics are labelled by their true poloidal mode number `m`, never by
block index, and amplitudes are peak-normalized because DCON's normalization
makes an absolute axis meaningless.

In [ ]:
if HAVE_RUN and ods is not None:
    import vaft.omas as vomas

    figure, axes = vomas.plot_mhd_linear_profile_displacement(ods)
    axes.set_title(f"{axes.get_title()}  --  VEST {SHOT} @ {TIME_MS} ms")
    plt.show()
else:
    print(
        "Would draw the displacement eigenfunction of the least-stable mode,\n"
        "one trace per poloidal harmonic, peak-normalized."
    )

In [ ]:
if HAVE_RUN and ods is not None:
    figure, axes = vomas.plot_mhd_linear_profile_b_field_perturbed(ods)
    plt.show()
else:
    print(
        "Would draw the perturbed field profile DCON derives from the same\n"
        "eigenfunction, as i(m - nq) xi.grad(psi)."
    )

In [ ]:
if HAVE_RUN and ods is not None:
    figure, axes = vomas.plot_mhd_linear_overview_eigenfunction(ods)
    plt.show()
else:
    print(
        "Would draw the overview panel for the least-stable mapped mode:\n"
        "displacement and perturbed field together."
    )

`plot_mhd_linear_time_energy_perturbed` draws the energy eigenvalue against
time, one trace per toroidal mode. It is included for completeness and is the
one plot here that a single time slice cannot really exercise -- it comes into
its own across a scan, where the marginal point shows up as a trace crossing
zero. With one slice it draws one point per mode.

In [ ]:
if HAVE_RUN and ods is not None:
    figure, axes = vomas.plot_mhd_linear_time_energy_perturbed(ods)
    for line in axes.lines:
        line.set_marker("o")
    axes.axhline(0.0, color="0.6", linewidth=0.8, zorder=0)
    plt.show()
    print("One time slice, so one point per mode. Zero is the marginal line.")
else:
    print("Would draw the energy eigenvalue against time, one trace per mode.")

## 6. What this run establishes

**It does establish** whether this reconstructed equilibrium is ideally stable
to the modes solved, and where the displacement of the least stable one sits
radially and in poloidal harmonic content.

**It does not establish** that VEST was stable at this moment. The input is an
EFIT reconstruction, and ideal stability is sensitive to the edge current and
pressure gradients a reconstruction constrains only weakly. A stability
conclusion that matters should be repeated across the reconstruction's own
uncertainty, not read off one equilibrium.

**It says nothing about resistive modes.** A DCON-stable equilibrium can still
be tearing unstable; that is RDCON's question, in
`linear_resistive_stability_analysis_with_rdcon.ipynb`, which starts from the
same prepared case.

**The eigenvalue is dimensionless.** Its sign and the ordering between modes
are meaningful; its magnitude in joules is not, whatever the IMAS field is
called.

## 7. Cleanup

The run directory holds the Euler-Lagrange intermediates, which are large and
regenerable -- `contour.bin` alone is about 50 MB per mode. The inputs that
produced them are in the repository, so re-running this notebook reproduces
everything. Comment out the `close()` to keep a run for inspection.

In [ ]:
if WORKDIR is not None and WORKDIR.exists():
    _size_mb = sum(p.stat().st_size for p in WORKDIR.rglob("*") if p.is_file()) / 1e6
    print(f"releasing {WORKDIR} ({_size_mb:.0f} MB)")
_cleanup.close()
print("done" if WORKDIR is None else f"removed: {not WORKDIR.exists()}")